# Self-Healing Machine Learning (SHML)

Automated Failure Detection, Diagnosis, Recovery,
and Safety Validation for Credit-Default Prediction


# Phase C — Controlled Missingness and Recovery

## Research Objective

Evaluate whether SHML's existing median-imputation
strategy maintains or improves predictive performance
compared with a predefined zero-imputation benchmark.

## Research Hypotheses

H1: Increasing BILL_AMT1 missingness produces measurable
predictive degradation under at least some conditions.

H2: Recovery effectiveness differs between imputation
strategies.

H4: The magnitude of performance changes depends on
missingness severity.

## Experimental Configuration

- Dataset: UCI Default of Credit Card Clients
- Model: Existing Logistic Regression baseline
- Corrupted feature: BILL_AMT1 only
- Missingness: 0%, 5%, 15%, 30%, 60%
- Random seed: 42 for the initial experiment
- Classification threshold: 0.50
- Healthy reference: Uncorrupted validation data
- Benchmark: Zero imputation
- SHML recovery: Training-median imputation
- Evaluation: F1, precision, recall, accuracy, ROC-AUC

## Experimental Controls

The model, training data, validation observations,
random missingness masks, and classification threshold
remain fixed between recovery strategies.

The test dataset remains untouched.

This initial experiment is exploratory. Statistical
validation across additional seeds follows in Phase D.


In [1]:

import numpy as np


def inject_bill_amt1_missingness(
    X_batch,
    missingness_rate,
    seed=42
):
    """
    Inject MCAR missingness into BILL_AMT1 only.

    Returns:
        corrupted DataFrame
        Boolean mask of modified rows
    """

    if not 0 <= missingness_rate <= 1:
        raise ValueError(
            "Missingness rate must be between 0 and 1."
        )

    if "BILL_AMT1" not in X_batch.columns:
        raise ValueError("BILL_AMT1 is missing.")

    X_corrupted = X_batch.copy(deep=True)

    rng = np.random.default_rng(seed)

    n_missing = int(
        len(X_corrupted) * missingness_rate
    )

    selected_positions = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    mask = np.zeros(
        len(X_corrupted),
        dtype=bool
    )

    mask[selected_positions] = True

    X_corrupted.iloc[
        selected_positions,
        X_corrupted.columns.get_loc("BILL_AMT1")
    ] = np.nan

    return X_corrupted, mask


In [3]:

from pathlib import Path
import sys

project_root = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "src" / "run_pipeline.py").exists()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.experiments.bill_amt1 import (
    inject_bill_amt1_missingness
)

import pandas as pd

# Lightweight function test.
# No model training or dataset loading needed.
sample = pd.DataFrame({
    "BILL_AMT1": range(100),
    "BILL_AMT2": range(100, 200)
})

corrupted, mask = inject_bill_amt1_missingness(
    sample,
    missingness_rate=0.15,
    seed=42
)

assert corrupted["BILL_AMT1"].isna().sum() == 15
assert corrupted["BILL_AMT2"].equals(sample["BILL_AMT2"])
assert sample.isna().sum().sum() == 0
assert mask.sum() == 15

print("Phase C missingness injection test passed!")


ModuleNotFoundError: No module named 'src.experiments'